# Step 7: volatility

- **Target:** how big day t's move is: the log of log(High / Low). A number, not up/down.
- **Models:**
  - **HAR:** linear regression on the average range over the last 1, 5 and 22 days. The standard simple model.
  - **Random forest** on the same three inputs.
  - **GRU** on the last 20 daily ranges, with three random seeds.
- **R2:** 1 is perfect, and 0 is no better than guessing the average of all earlier days.
- Same six walk-forward folds as before.

In [ ]:
from functools import partial

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

from data import load_all
from features import log_range, make_har, make_range_sequence
from gru import HIDDEN_SIZES
from volatility import (fit_gru_vol, fit_regressor, make_vol_dataset, pick_by_mse, pick_gru_vol_setting,
                        vol_scores, walk_forward_vol)

markets = load_all()

## Is there structure to learn?

How much today resembles the recent past (correlation with 1, 5 and 22 days earlier). Direction has almost none; the range has plenty.

In [ ]:
rows = {}
for market, df in markets.items():
    up = (df['ret'] > 0).astype(float)
    size = log_range(df)
    rows[market] = {
        "direction, 1 day": up.autocorr(1),
        "range, 1 day": size.autocorr(1),
        "range, 5 days": size.autocorr(5),
        "range, 22 days": size.autocorr(22),
    }
pd.DataFrame(rows).T.round(2)

## Run the models

The markets run in parallel. About 30 minutes, mostly the GRU.

In [ ]:
def run_all_markets(fit, settings, make_x, pick=pick_by_mse):
    runs = Parallel(n_jobs=6)(
        delayed(walk_forward_vol)(df, fit, settings, make_x, pick) for df in markets.values()
    )
    return dict(zip(markets, runs))

FOREST = [RandomForestRegressor(n_estimators=300, min_samples_leaf=leaf, random_state=0) for leaf in [10, 50, 200]]

har = run_all_markets(fit_regressor, [LinearRegression()], make_har)
forest = run_all_markets(fit_regressor, FOREST, make_har)
gru = {seed: run_all_markets(partial(fit_gru_vol, seed=seed), HIDDEN_SIZES, make_range_sequence,
                             partial(pick_gru_vol_setting, seed=seed))
       for seed in [0, 1, 2]}

## All six test years pooled

**p vs HAR** is a paired test of squared errors on the same days (Diebold-Mariano).

In [ ]:
def r2_of(guess, preds):
    """R2 of any guess, against the average of all earlier days."""
    error = guess - preds["actual"]
    error_of_average = preds["average_so_far"] - preds["actual"]
    return 1 - (error ** 2).sum() / (error_of_average ** 2).sum()

rows = {}
for market in markets:
    h, f, g = har[market], forest[market], gru[0][market]
    baseline = h["predicted"]
    rows[market] = {
        "same as yesterday R2": r2_of(h["same_as_yesterday"], h),
        "HAR R2": r2_of(h["predicted"], h),
        "forest R2": r2_of(f["predicted"], f),
        "GRU R2": r2_of(g["predicted"], g),
        "p forest vs HAR": vol_scores(f, baseline)["p vs baseline"],
        "p GRU vs HAR": vol_scores(g, baseline)["p vs baseline"],
        "p HAR vs same as yesterday": vol_scores(h, h["same_as_yesterday"])["p vs baseline"],
    }
pd.DataFrame(rows).T.round(3)

## Do the GRU's results depend on the seed?

In [ ]:
rows = {}
for market in markets:
    for seed in gru:
        g = gru[seed][market]
        rows[market, seed] = {
            "GRU R2": r2_of(g["predicted"], g),
            "p vs HAR": vol_scores(g, har[market]["predicted"])["p vs baseline"],
        }
pd.DataFrame(rows).T.round(3).unstack()

## Year by year

HAR's R2 in each test year, and how many of the 36 market-years each model beats HAR.

In [ ]:
def yearly_r2(preds_by_market):
    return pd.DataFrame({
        market: {year: r2_of(f["predicted"], f) for year, f in p.groupby("year")}
        for market, p in preds_by_market.items()
    }).T

har_years = yearly_r2(har)
display(har_years.round(2))
pd.Series({
    "forest": (yearly_r2(forest) > har_years).sum().sum(),
    "GRU, seed 0": (yearly_r2(gru[0]) > har_years).sum().sum(),
    "GRU, seed 1": (yearly_r2(gru[1]) > har_years).sum().sum(),
    "GRU, seed 2": (yearly_r2(gru[2]) > har_years).sum().sum(),
}, name="market-years above HAR, out of 36").to_frame()

## What is the model paying attention to?

HAR's weights, fitted on 2012-2022. Because it's linear you can read them directly: a larger weight means that period counts more.

In [ ]:
weights = {}
for market, df in markets.items():
    X, y = make_vol_dataset(df, make_har)
    keep = X.index.year <= 2022
    model = LinearRegression().fit(X[keep], y[keep])
    weights[market] = dict(zip(X.columns, model.coef_))
pd.DataFrame(weights).T.round(2)

## Data check: Indonesia's fake opens

In 2020, Indonesia's fake-open days have bigger ranges, so those Highs and Lows may be affected. HAR's R2 with those days left out of the scoring:

In [ ]:
p = har["Indonesia"]
df = markets["Indonesia"]
fake_open = (df["Open"] == df["Close"].shift(1))[p.index]
print(f"all test days: {r2_of(p['predicted'], p):.3f} ({len(p)} days)")
real = p[~fake_open]
print(f"real opens only: {r2_of(real['predicted'], real):.3f} ({len(real)} days)")